# Finetune a small LLM for German spell checking

In this notebook we finetune a small instruction tuned LLM (**Llama 3.2 1B**) with [Unsloth](https://unsloth.ai/) so that it corrects spelling mistakes in German text.

The whole run fits on a **free Tesla T4 Google Colab instance**: press "*Runtime*" and then "*Run all*".

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), how to save it and how to run it with [Ollama](#Ollama).

This notebook is based on Unsloth's [Llama3 (8B) Ollama template](https://github.com/unslothai/notebooks) (LGPL-3.0). To install Unsloth on your own machine, follow [their guide](https://unsloth.ai/docs/get-started/install).

### Installation

Running this on your own machine? Use Python 3.12 or 3.13 - the pinned `datasets` version does not work on Python 3.14 yet.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Load the model

We use `unsloth/Llama-3.2-1B-Instruct`: it is small enough to finetune in a few minutes on a T4, German is one of its officially supported languages and - being an *Instruct* model - it already knows how to follow instructions.

Because the model is small we load it in 16 bit instead of 4 bit: that is a little more accurate, trains faster and still fits easily into the memory of a T4.

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024 # Our examples are short: 99% of the input + output pairs have fewer than 300 tokens.
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # 4 bit quantization saves memory - a 1B model does not need it, 16 bit is faster and more accurate.

# Other small models you could try. Note: the chat template below is Llama specific,
# if you pick a non-Llama model you have to adapt it (see the "Chat template" section).
small_models = [
    "unsloth/Llama-3.2-1B-Instruct",  # our choice: 1.2B parameters, German officially supported
    "unsloth/Llama-3.2-3B-Instruct",  # bigger sibling, better quality, about 3x slower
    "unsloth/Qwen2.5-0.5B-Instruct",  # even smaller, decent German, uses the ChatML format
    "unsloth/gemma-3-1b-it",          # careful: Gemma 3 needs float32 on a T4 and is slow there
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

<a name="Data"></a>
### Data Prep

Our dataset is a CSV file with 10,000 German sentences. The column `text` contains a sentence with (synthetic) spelling mistakes, the column `summary` the correct version. We load it directly from the workshop's GitHub repository, so this notebook also runs in Colab without uploading anything.

We keep 200 examples aside as a test set - later we use them to check what the model has learned.

In [ ]:
from datasets import load_dataset

csv_url = "https://raw.githubusercontent.com/oliverguhr/workshop-ki-deepdive/main/data/fix.spelling.mini.csv"
# csv_url = "./data/fix.spelling.mini.csv" # use this if you run the notebook locally inside the workshop repository

dataset = load_dataset("csv", data_files = csv_url, split = "train")
dataset = dataset.rename_columns({"text": "input", "summary": "output"})
dataset = dataset.train_test_split(test_size = 200, seed = 3407)
train_dataset, test_dataset = dataset["train"], dataset["test"]
print(train_dataset)
train_dataset[0]

For `Ollama` and `llama.cpp` to work like a chatbot, a training example must consist of exactly two parts: what the user says and what the model answers.

Unsloth's `to_sharegpt` builds these pairs. `merged_prompt` is the user message - here it is simply the misspelled sentence (the instruction "please correct this" goes into the system prompt in the next step, so that later on you can just paste a sentence into Ollama). `output_column_name` is the answer the model should learn.

Spell checking is a single turn task, so we set `conversation_extension = 1`. Unsloth's template uses `3` to glue random examples together into fake multi-turn chats - we do not want that here.

In [ ]:
from unsloth import to_sharegpt, standardize_sharegpt

train_dataset = to_sharegpt(
    train_dataset,
    merged_prompt = "{input}",
    output_column_name = "output",
    conversation_extension = 1, # 1 = single turn. No fake multi-turn chats for spell checking.
)
train_dataset = standardize_sharegpt(train_dataset)
train_dataset[0]["conversations"]

### Chat template

Now we define how a training example looks as plain text. We use the **Llama 3 instruct format**, because `Llama-3.2-1B-Instruct` was trained with it. Unsloth fills in `{SYSTEM}`, `{INPUT}` and `{OUTPUT}`.

The system prompt tells the model what its job is. Later we also put it into the Ollama `Modelfile`, so in Ollama you can simply paste a sentence and get the correction back.

If you switch to a non-Llama model, use its own format here. For example the ChatML format used by Qwen:

```python
chat_template = """<|im_start|>system
{SYSTEM}<|im_end|>
<|im_start|>user
{INPUT}<|im_end|>
<|im_start|>assistant
{OUTPUT}<|im_end|>"""
```

In [ ]:
from unsloth import apply_chat_template

system_prompt = "Du bist ein Korrekturprogramm. Korrigiere alle Rechtschreib- und Grammatikfehler im Text des Nutzers und antworte nur mit dem korrigierten Text."

chat_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{SYSTEM}<|eot_id|><|start_header_id|>user<|end_header_id|>

{INPUT}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{OUTPUT}<|eot_id|>"""

train_dataset = apply_chat_template(
    train_dataset,
    tokenizer = tokenizer,
    chat_template = chat_template,
    default_system_message = system_prompt,
)
print(train_dataset[0]["text"])

### Before training

Let's see how the model does *before* finetuning. The freshly added LoRA adapters are initialised with zeros, so right now the model still behaves exactly like the original `Llama-3.2-1B-Instruct`.

`korrigiere()` sends a sentence through the chat template and lets the model generate a correction. We use greedy decoding (`do_sample = False`): for spell checking we want the most likely correction, not a creative one.

We look at four test sentences with typical typos (swapped, missing and doubled letters, wrong capitalisation). The dataset also contains a harder kind of noise where every `e` became an `ä` - have a look at `test_dataset[0]` and `test_dataset[2]` if you want to see how the model copes with that.

In [ ]:
def korrigiere(text, max_new_tokens = 256):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = True)
    inputs = tokenizer(prompt, return_tensors = "pt", add_special_tokens = False).to("cuda")
    output_ids = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        do_sample = False, # greedy decoding: always take the most likely token
        temperature = 1.0, top_p = 1.0, # keeps transformers from warning about unused sampling settings
        pad_token_id = tokenizer.eos_token_id,
    )
    return tokenizer.decode(output_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens = True).strip()

def zeige_beispiele(examples):
    for example in examples:
        print("Input:     ", example["input"])
        print("Correction:", korrigiere(example["input"]))
        print("Expected:  ", example["output"])
        print("-" * 100)

test_examples = test_dataset.select([1, 3, 6, 7])
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
zeige_beispiele(test_examples)

<a name="Train"></a>
### Train the model

Now let's train our model. `max_steps = 300` with a batch size of 8 means the model sees 2,400 examples - enough to learn the task and done in a few minutes on a T4. For a full run over all 9,800 training examples set `num_train_epochs = 1` and remove `max_steps`.

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
FastLanguageModel.for_training(model) # switch back from inference to training mode
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 1, # Use GA to mimic a larger batch size
        warmup_steps = 5,
        max_steps = 300,
        # num_train_epochs = 1, # For a full training run - then remove max_steps!
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

We only want to train on the model's *answers*, not on the misspelled input - otherwise the model would also learn to *produce* spelling mistakes. Unsloth's `train_on_responses_only` masks everything up to the assistant header, so those tokens do not contribute to the loss. It already knows where the user and assistant parts start from the chat template we applied above.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer) # Unsloth knows the user/assistant markers from our chat template

We verify masking is actually done. First the full example:

In [ ]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

And now only the part the model is trained on - the system prompt and the misspelled input are masked out:

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[0]["labels"]])

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### After training

The same four test sentences and the same function as above - but now with the trained LoRA adapters.

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
zeige_beispiele(test_examples)

Try your own sentence:

In [ ]:
print(korrigiere("Das ist ein Satz mit vielen Rechtschreibfelern, den das Modell korigieren sol."))

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to GGUF for Ollama, scroll down!

In [ ]:
model.save_pretrained("llama-3.2-1b-german-spelling-lora")  # Local saving
tokenizer.save_pretrained("llama-3.2-1b-german-spelling-lora")
# model.push_to_hub("your_name/llama-3.2-1b-german-spelling-lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama-3.2-1b-german-spelling-lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "llama-3.2-1b-german-spelling-lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference
    print(korrigiere(test_examples[0]["input"]))

<a name="Ollama"></a>
### Ollama Support

[Unsloth](https://github.com/unslothai/unsloth) can export the finetuned model to GGUF and automatically create a [Modelfile](https://github.com/ollama/ollama/blob/main/docs/modelfile.md) for [Ollama](https://ollama.com/). That gives us a spell checker we can run locally on a laptop.

Let's first install `Ollama`!

In [ ]:
# ollama's installer extracts a zstd archive and refuses to run without zstd, which Colab does not ship
!command -v zstd >/dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y zstd) >/dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

Next, we save the model to GGUF / llama.cpp. Unsloth merges the LoRA adapters into the model, downloads `llama.cpp` and converts the model. We use `q8_0` (8 bit) - for a 1B model that is only about 1.3 GB.

This creates two folders: `llama-3.2-1b-german-spelling` with the merged 16 bit model in Hugging Face format and `llama-3.2-1b-german-spelling_gguf` with the `.gguf` file and an Ollama `Modelfile`.

Other quantizations like `q4_k_m` are possible too, see the [Unsloth docs](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf).

In [ ]:
# Save to 8bit Q8_0
if True: model.save_pretrained_gguf("llama-3.2-1b-german-spelling", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/llama-3.2-1b-german-spelling", tokenizer, token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF (smaller, a little less accurate)
if False: model.save_pretrained_gguf("llama-3.2-1b-german-spelling", tokenizer, quantization_method = "q4_k_m")

We use `subprocess` to start `Ollama` in a non blocking fashion! On your own computer you can simply open a new terminal and type `ollama serve`, but in Colab we have to use this hack.

In [ ]:
import subprocess
import time

import requests

subprocess.Popen(["ollama", "serve"])

# Wait until Ollama is actually answering instead of guessing how long it needs.
for _ in range(60):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout = 2).ok:
            print("Ollama is ready!")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError(
        "Ollama did not become ready on http://localhost:11434 within 60s. "
        "Check the `ollama serve` output above."
    )

`Ollama` needs a `Modelfile`, which specifies the model file, the prompt format, the system prompt and sampling parameters. Unsloth generated one for us with the correct Llama 3 prompt format, but it is meant for chatbots: it sets a high sampling temperature and does not contain our system prompt.

So we make two changes: we add our system prompt with `SYSTEM` (that is why you can just paste a sentence into Ollama later) and we set `temperature 0`, because a spell checker should be deterministic.

In [ ]:
modelfile_path = "llama-3.2-1b-german-spelling_gguf/Modelfile"
with open(modelfile_path) as f:
    lines = [line for line in f.read().splitlines() if not line.startswith(("PARAMETER temperature", "PARAMETER min_p", "PARAMETER top_p", "SYSTEM"))]
lines.append(f'SYSTEM """{system_prompt}"""')
lines.append("PARAMETER temperature 0")
modelfile = "\n".join(lines) + "\n"
with open(modelfile_path, "w") as f:
    f.write(modelfile)
print(modelfile)

We now create an `Ollama` model called `german-spelling` from the `Modelfile`:

In [ ]:
!ollama create german-spelling -f ./llama-3.2-1b-german-spelling_gguf/Modelfile

And now we can do inference on it via the `Ollama` API! The system prompt is part of the Modelfile, so we only send the misspelled sentence.

You can also upload the model to `Ollama` and try the `Ollama` Desktop app, see https://www.ollama.com/

In [ ]:
response = requests.post(
    "http://localhost:11434/api/chat",
    json = {
        "model": "german-spelling",
        "messages": [{"role": "user", "content": test_examples[0]["input"]}],
        "stream": False,
    },
    timeout = 300,
)
print("Input:     ", test_examples[0]["input"])
print("Correction:", response.json()["message"]["content"])
print("Expected:  ", test_examples[0]["output"])

# Interactive mode

### ⭐ To use the spell checker interactively, first click the **| >_ |** button to open a terminal.
![](https://raw.githubusercontent.com/unslothai/unsloth/nightly/images/Where_Terminal.png)

### ⭐ Then type `ollama run german-spelling`, paste any German sentence with typos and press `ENTER`. To exit, hit `CTRL + D`.

On your own computer: copy the `.gguf` file and the `Modelfile` from the `llama-3.2-1b-german-spelling_gguf` folder, run `ollama create german-spelling -f Modelfile` and then `ollama run german-spelling`.

---

This notebook is based on the Unsloth notebooks, which are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme). If you have questions about Unsloth, there is a [Discord](https://discord.gg/unsloth) channel, an [Installation Guide](https://unsloth.ai/docs/get-started/install) and a [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms).